In [1]:
# Show GPU info early
try:
    import torch
    if torch.cuda.is_available():
        idx = torch.cuda.current_device()
        print('GPU:', torch.cuda.get_device_name(idx))
        print('CUDA:', torch.version.cuda)
        print('VRAM (GB):', round(torch.cuda.get_device_properties(idx).total_memory / 1e9, 2))
    else:
        print('GPU: not available')
except Exception as e:
    print('GPU check failed:', e)

GPU: NVIDIA A100-SXM4-80GB
CUDA: 12.8
VRAM (GB): 85.09


In [2]:
%%capture
import os, re
is_colab = "COLAB_" in "".join(os.environ.keys())
if not is_colab:
    # Local: keep it simple; install the basics.
    %pip -q install unsloth transformers datasets accelerate bitsandbytes trl peft pillow scikit-learn pandas
else:
    # Colab: follow Unsloth's recommended install pattern (more reliable).
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10":"0.0.34","2.9":"0.0.33.post1","2.8":"0.0.32.post2"}.get(v, "0.0.34")
    !pip -q install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip -q install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip -q install "transformers==4.56.2"
    !pip -q install --no-deps "trl==0.22.2"
    !pip -q install "pillow<11.0"
    !pip -q install scikit-learn pandas

In [3]:
import os
from pathlib import Path

# Colab-style auto prep (Drive + unzip) like RX/RX_BASE_Vision_Finetune.ipynb
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ---- Data location knobs ----
# If you're on Colab: upload these into Google Drive folder below:
#   - train_data.json
#   - (optional) test_data.json
#   - RX_BASE_Imagenes.zip  (contains Imagenes/*.jpg)
DRIVE_FOLDER = os.environ.get('RX_DRIVE_FOLDER', '/content/drive/MyDrive/Internship/RX_BASE_Finetune')
ZIP_FILENAME = os.environ.get('RX_ZIP_FILENAME', 'RX_BASE_Imagenes.zip')

# Where images end up after unzip (Colab default)
IMAGES_DIR = Path(os.environ.get('RX_IMAGES_DIR', '/content/images')).expanduser()
TRAIN_JSON = Path(os.environ.get('RX_TRAIN_JSON', f'{DRIVE_FOLDER}/train_data.json')).expanduser()
TEST_JSON = Path(os.environ.get('RX_TEST_JSON', f'{DRIVE_FOLDER}/test_data.json')).expanduser()

# Output style
# - 'full'      : train on the full assistant response (e.g. 'Diorite. This is a ...')
# - 'label_only': train to output ONLY the rock name (e.g. 'Diorite')
ANSWER_STYLE = os.environ.get('RX_ANSWER_STYLE', 'label_only').strip().lower()

# Training knobs
MODEL_NAME = os.environ.get('RX_MODEL', 'unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit')
SUBSET = int(os.environ.get('RX_SUBSET', '0'))  # set 0 for full
MAX_STEPS = int(os.environ.get('RX_MAX_STEPS', '350'))
BATCH_SIZE = int(os.environ.get('RX_BATCH_SIZE', '2'))
GRAD_ACCUM = int(os.environ.get('RX_GRAD_ACCUM', '4'))
EPOCHS = int(os.environ.get('RX_EPOCHS', '3'))  # 0 disables epoch-based steps

# Evaluation knobs
EVAL_SIZE = int(os.environ.get('RX_EVAL_SIZE', '100'))  # 0 = evaluate on all eval examples
EVAL_TEMPERATURE = float(os.environ.get('RX_EVAL_TEMPERATURE', '0.05'))

print('IN_COLAB:', IN_COLAB)
print('DRIVE_FOLDER:', DRIVE_FOLDER)
print('ZIP_FILENAME:', ZIP_FILENAME)
print('IMAGES_DIR:', IMAGES_DIR)
print('TRAIN_JSON:', TRAIN_JSON)
print('TEST_JSON:', TEST_JSON)
print('MODEL_NAME:', MODEL_NAME)
print('ANSWER_STYLE:', ANSWER_STYLE)
print('EVAL_SIZE:', EVAL_SIZE)
print('EPOCHS:', EPOCHS)

# Friendly warning: huge non-4bit models will offload + create meta tensors, which breaks Unsloth training.
if ('90B' in MODEL_NAME or '27B' in MODEL_NAME) and (('4bit' not in MODEL_NAME) and ('bnb-4bit' not in MODEL_NAME)):
    print("WARNING: You selected a large model without *-bnb-4bit. This commonly triggers CPU offload + meta tensors and training will fail.")
    print("         Use a *-bnb-4bit checkpoint name (e.g. unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit) or pick a smaller model.")

# ---- Execute auto-prep on Colab ----
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    if not os.path.exists(DRIVE_FOLDER):
        raise FileNotFoundError(
            f"Drive folder not found: {DRIVE_FOLDER}. "
            "Create it and upload train_data.json (+ optionally test_data.json) and RX_BASE_Imagenes.zip."
        )
    else:
        print('Drive contents:', os.listdir(DRIVE_FOLDER))

    zip_path = f"{DRIVE_FOLDER}/{ZIP_FILENAME}"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Zip not found: {zip_path}")

    # Unzip if needed
    if (not IMAGES_DIR.exists()) or (IMAGES_DIR.exists() and len(list(IMAGES_DIR.glob('*.jpg'))) < 100):
        print('📦 Extracting images...')
        # Unzip into /content, then move Imagenes -> /content/images (same as old notebook)
        !unzip -q "{zip_path}" -d /content/
        !mkdir -p /content/images
        !mv /content/Imagenes/* /content/images/ 2>/dev/null || true
        print('✅ Extracted images into', IMAGES_DIR)
    else:
        print('✅ Images already present:', len(list(IMAGES_DIR.glob('*.jpg'))))

IN_COLAB: True
DRIVE_FOLDER: /content/drive/MyDrive/Internship/RX_BASE_Finetune
ZIP_FILENAME: RX_BASE_Imagenes.zip
IMAGES_DIR: /content/images
TRAIN_JSON: /content/drive/MyDrive/Internship/RX_BASE_Finetune/train_data.json
TEST_JSON: /content/drive/MyDrive/Internship/RX_BASE_Finetune/test_data.json
MODEL_NAME: unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit
ANSWER_STYLE: label_only
EVAL_SIZE: 100
EPOCHS: 3
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive contents: ['dataset_metadata.json', 'finetune_colab.ipynb', 'finetune_vision_llm.py', 'prepare_dataset.py', 'RX_BASE_Imagenes.zip', 'RX_BASE_Vision_Finetune_A100.ipynb', 'test_data.json', 'train_alpaca.json', 'train_data.json', 'RX_BASE_Vision_Finetune.ipynb', 'RX_BASE_Unsloth_Vision_Finetune.ipynb', 'Models']
✅ Images already present: 5600


In [4]:
import json
from pathlib import Path

with open(TRAIN_JSON, 'r', encoding='utf-8') as f:
    raw = json.load(f)

print('Loaded examples:', len(raw))
print('First keys:', list(raw[0].keys()))
print('First image field:', raw[0].get('image'))

Loaded examples: 4480
First keys: ['image', 'conversations']
First image field: /Users/okis/CascadeProjects/meteorite-detection-network/finetune/../RX_BASE/Imagenes/104_01311.jpg


In [5]:
# Convert RX -> Unsloth vision messages format (same as successful notebooks)
from pathlib import Path

def _to_label_only(text: str) -> str:
    # RX assistant text is typically: "Diorite. This is ..."
    # Keep the rock name only (before the first period)
    rock = text.split('.', 1)[0].strip()
    return rock if rock else text.strip()

def convert_rx_to_messages(ex, images_dir: Path):
    image_name = Path(str(ex['image'])).name
    image_path = images_dir / image_name

    convs = ex['conversations']
    user_text = str(convs[0].get('value', '')).replace('<image>\n', '').replace('<image>', '').strip()
    assistant_text = str(convs[1].get('value', '')).strip()

    if ANSWER_STYLE == 'label_only':
        assistant_text = _to_label_only(assistant_text)
        # Make the instruction explicit for the model
        user_text = user_text + " Reply with only the rock type name."

    return {
        'messages': [
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': str(image_path)},
                    {'type': 'text', 'text': user_text},
                ],
            },
            {
                'role': 'assistant',
                'content': [
                    {'type': 'text', 'text': assistant_text},
                ],
            },
        ]
    }

data = raw if SUBSET == 0 else raw[:min(SUBSET, len(raw))]
dataset = [convert_rx_to_messages(ex, IMAGES_DIR) for ex in data]
print('Converted examples:', len(dataset))
print('Example 0:', dataset[0])

Converted examples: 4480
Example 0: {'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/content/images/104_01311.jpg'}, {'type': 'text', 'text': 'Classify this petrographic thin section image. What type of rock is this? Reply with only the rock type name.'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': 'Diorite'}]}]}


In [6]:
# Epoch-based step calculator (optional)
import math

if EPOCHS > 0:
    global_batch = max(1, BATCH_SIZE * GRAD_ACCUM)
    steps_per_epoch = math.ceil(len(dataset) / global_batch)
    MAX_STEPS = max(1, steps_per_epoch * EPOCHS)
    print('Epoch-based steps enabled')
    print('Global batch:', global_batch)
    print('Steps per epoch:', steps_per_epoch)
    print('Total steps (MAX_STEPS):', MAX_STEPS)
else:
    print('Epoch-based steps disabled; using MAX_STEPS:', MAX_STEPS)

Epoch-based steps enabled
Global batch: 8
Steps per epoch: 560
Total steps (MAX_STEPS): 1680


In [7]:
# Load model + tokenizer (Unsloth Vision)
import torch
from unsloth import FastVisionModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template

# Heuristic: if you want 4bit, use an Unsloth *-bnb-4bit checkpoint name.
# Loading a huge model (e.g. 90B) in full precision will trigger CPU offload + meta tensors and break training.
LOAD_IN_4BIT = ('4bit' in MODEL_NAME) or ('bnb-4bit' in MODEL_NAME)

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=LOAD_IN_4BIT,
    use_gradient_checkpointing='unsloth',
)

# Guard: fail fast if any parameters are still on the meta device (offload/dispatch incomplete).
meta_params = [n for n, p in model.named_parameters() if getattr(p, 'is_meta', False)]
if meta_params:
    example = meta_params[0]
    raise RuntimeError(
        "Model has meta tensors (parameters not materialized), so Unsloth training will fail.\n"
        f"Example meta param: {example}\n\n"
        "Fix: use a quantized checkpoint (recommended: *-bnb-4bit) or a smaller model.\n"
        "E.g. set RX_MODEL to: 'unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit' or 'unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit'."
    )

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    random_state=3407,
)

print('LOAD_IN_4BIT:', LOAD_IN_4BIT)
print('BF16 supported:', is_bfloat16_supported())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Mllama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Unsloth: Making `model.base_model.model.model.vision_model.transformer` require gradients
LOAD_IN_4BIT: True
BF16 supported: True


In [8]:
# Train
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator
from unsloth import is_bfloat16_supported
from unsloth import FastVisionModel

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        max_steps=MAX_STEPS,
        learning_rate=1e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=3407,
        output_dir='outputs',
        report_to='none',
        remove_unused_columns=False,
        dataset_text_field='',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_length=2048,
        save_strategy='steps',
        save_steps=max(50, MAX_STEPS // 4),
    ),
)

trainer_stats = trainer.train()
trainer_stats

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,480 | Num Epochs = 3 | Total steps = 1,680
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 134,348,800 of 10,804,569,635 (1.24% trained)


Step,Training Loss
25,2.635900
50,1.775300
75,0.921500
100,0.497100
125,0.225000
150,0.226700
175,0.159100
200,0.124900
225,0.158200
250,0.475400


TrainOutput(global_step=1680, training_loss=0.13886637808533298, metrics={'train_runtime': 9100.9754, 'train_samples_per_second': 1.477, 'train_steps_per_second': 0.185, 'total_flos': 3.2822558660171016e+16, 'train_loss': 0.13886637808533298, 'epoch': 3.0})

In [9]:
# Quick inference smoke test (after training)
from PIL import Image
from unsloth import FastVisionModel
from transformers import TextStreamer

FastVisionModel.for_inference(model)

img_path = dataset[0]['messages'][0]['content'][0]['image']
prompt = dataset[0]['messages'][0]['content'][1]['text']

img = Image.open(img_path).convert('RGB')

messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': prompt},
        ],
    }
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(img, input_text, add_special_tokens=False, return_tensors='pt').to('cuda')

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=32 if ANSWER_STYLE == 'label_only' else 96,
    temperature=0.1,
    use_cache=False,
    min_p=0.1,
 )

Diorite<|eot_id|>


In [10]:
# Evaluate on held-out set with classification metrics
import json
import re
import random
from typing import List

import pandas as pd
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    )
from unsloth import FastVisionModel

def _extract_label(text: str) -> str:
    # Robust label extractor for both 'label_only' and 'full' styles.
    # Examples:
    # - "Diorite" -> Diorite
    # - "Diorite. This is ..." -> Diorite
    # - "Diorite\n" -> Diorite
    if text is None:
        return ""
    t = str(text).strip()
    if not t:
        return ""
    t = t.splitlines()[0].strip()
    t = t.split(".", 1)[0].strip()
    t = re.sub(r"[^A-Za-z0-9 _\-]+", "", t).strip()
    return t

def _build_eval_raw(raw_train: list) -> list:
    # Prefer explicit test file if provided; otherwise, take a deterministic 20% split from train_data.json
    if TEST_JSON.exists():
        with open(TEST_JSON, 'r', encoding='utf-8') as f:
            return json.load(f)
    rng = random.Random(3407)
    idxs = list(range(len(raw_train)))
    rng.shuffle(idxs)
    n_eval = max(1, int(0.2 * len(idxs)))
    return [raw_train[i] for i in idxs[:n_eval]]

eval_raw = _build_eval_raw(raw)
eval_data = eval_raw if EVAL_SIZE == 0 else eval_raw[:min(EVAL_SIZE, len(eval_raw))]
eval_dataset = [convert_rx_to_messages(ex, IMAGES_DIR) for ex in eval_data]
print(f"Eval examples: {len(eval_dataset)} (source: {'TEST_JSON' if TEST_JSON.exists() else '20% split from TRAIN_JSON'})")

# Generate predictions
FastVisionModel.for_inference(model)

y_true: List[str] = []
y_pred: List[str] = []
unknown_preds = 0

for item in eval_dataset:
    img_path = item['messages'][0]['content'][0]['image']
    prompt = item['messages'][0]['content'][1]['text']
    true_text = item['messages'][1]['content'][0]['text']

    img = Image.open(img_path).convert('RGB')
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt},
            ],
        }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(img, input_text, add_special_tokens=False, return_tensors='pt').to('cuda')
    input_len = inputs['input_ids'].shape[-1]

    out = model.generate(
        **inputs,
        max_new_tokens=32 if ANSWER_STYLE == 'label_only' else 96,
        temperature=EVAL_TEMPERATURE,
        use_cache=False,
        min_p=0.1,
    )
    new_tokens = out[0][input_len:]
    pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    yt = _extract_label(true_text)
    yp = _extract_label(pred_text)
    if not yp:
        unknown_preds += 1
        yp = "<EMPTY>"
    y_true.append(yt)
    y_pred.append(yp)

labels_true = sorted(set(y_true))
extra_labels = sorted(set(y_pred) - set(labels_true))
labels_cm = labels_true + extra_labels

acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, labels=labels_true, average='macro', zero_division=0)
f1_weighted = f1_score(y_true, y_pred, labels=labels_true, average='weighted', zero_division=0)
prec_macro = precision_score(y_true, y_pred, labels=labels_true, average='macro', zero_division=0)
rec_macro = recall_score(y_true, y_pred, labels=labels_true, average='macro', zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=labels_cm)

print('Accuracy:', round(acc, 4))
print('F1 macro:', round(f1_macro, 4), ' | F1 weighted:', round(f1_weighted, 4))
print('Precision macro:', round(prec_macro, 4), ' | Recall macro:', round(rec_macro, 4))
print('Empty/unknown predictions:', unknown_preds)

print('\nPer-class report (true labels):')
print(classification_report(y_true, y_pred, labels=labels_true, zero_division=0))

cm_df = pd.DataFrame(
    cm,
    index=[f"true:{l}" for l in labels_cm],
    columns=[f"pred:{l}" for l in labels_cm],
)
cm_df

Eval examples: 100 (source: TEST_JSON)
Accuracy: 0.93
F1 macro: 0.9026  | F1 weighted: 0.9264
Precision macro: 0.942  | Recall macro: 0.8915
Empty/unknown predictions: 0

Per-class report (true labels):
              precision    recall  f1-score   support

    Andesite       0.93      0.93      0.93        15
      Basalt       1.00      0.83      0.91         6
     Diorite       1.00      1.00      1.00         8
      Gabbro       1.00      1.00      1.00         4
      Gneiss       0.86      0.86      0.86         7
     Granite       1.00      0.50      0.67         4
   Limestone       0.93      1.00      0.96        13
      Marble       1.00      1.00      1.00         6
  Peridotite       1.00      1.00      1.00         9
    Phyllite       0.83      1.00      0.91         5
    Rhyolite       0.78      1.00      0.88         7
   Sandstone       0.86      0.86      0.86         7
      Schist       1.00      0.50      0.67         2
       Shale       1.00      1.00      1

,pred:Andesite,pred:Basalt,pred:Diorite,pred:Gabbro,pred:Gneiss,pred:Granite,pred:Limestone,pred:Marble,pred:Peridotite,pred:Phyllite,pred:Rhyolite,pred:Sandstone,pred:Schist,pred:Shale
true:Andesite,14,0,0,0,0,0,0,0,0,0,1,0,0,0
true:Basalt,1,5,0,0,0,0,0,0,0,0,0,0,0,0
true:Diorite,0,0,8,0,0,0,0,0,0,0,0,0,0,0
true:Gabbro,0,0,0,4,0,0,0,0,0,0,0,0,0,0
true:Gneiss,0,0,0,0,6,0,1,0,0,0,0,0,0,0
true:Granite,0,0,0,0,1,2,0,0,0,0,0,1,0,0
true:Limestone,0,0,0,0,0,0,13,0,0,0,0,0,0,0
true:Marble,0,0,0,0,0,0,0,6,0,0,0,0,0,0
true:Peridotite,0,0,0,0,0,0,0,0,9,0,0,0,0,0
true:Phyllite,0,0,0,0,0,0,0,0,0,5,0,0,0,0


In [11]:
# Save LoRA adapters (and copy to Drive on Colab)
import shutil
from pathlib import Path
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path(f'rx_lora_{timestamp}')
model.save_pretrained(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))
print(f"Saved LoRA adapters to ./{OUT_DIR}")

# If on Colab, also copy to Google Drive folder
if 'IN_COLAB' in globals() and IN_COLAB:
    drive_target = Path(f"{DRIVE_FOLDER}/Models") / OUT_DIR.name
    print(f"Copying to Drive: {drive_target}")
    drive_target.parent.mkdir(parents=True, exist_ok=True)
    # Remove existing target to avoid copytree errors
    if drive_target.exists():
        shutil.rmtree(drive_target)
    shutil.copytree(OUT_DIR, drive_target)
    print(f"✅ Copied to Drive: {drive_target}")
else:
    print("Not running on Colab; skipping Drive copy.")

Saved LoRA adapters to ./rx_lora_20260226_004614
Copying to Drive: /content/drive/MyDrive/Internship/RX_BASE_Finetune/Models/rx_lora_20260226_004614
✅ Copied to Drive: /content/drive/MyDrive/Internship/RX_BASE_Finetune/Models/rx_lora_20260226_004614


In [ ]:
# Kill Colab runtime to prevent idle charges (run this manually after you're done)
if 'IN_COLAB' in globals() and IN_COLAB:
    print('Killing Colab runtime in 60 seconds...')
    import time
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print('Not on Colab; skipping runtime kill.')